# Feature Selection
The purpose of this script is to isolate the features to be used in my model for training, validation, and test

In [ ]:
import pandas as pd
import numpy as np
import os

DATA_PATH = "./data/"

In [ ]:
visits_df = pd.read_csv(os.path.join(DATA_PATH, "visits.csv"))
meds_df = pd.read_csv(os.path.join(DATA_PATH, "meds.csv"))
pmh_df = pd.read_csv(os.path.join(DATA_PATH, "pmh.csv"))

In [ ]:
split = {
    "random": {
        "train": pd.read_csv(os.path.join(DATA_PATH, "split_random_train.csv"), header=None),
        "val": pd.read_csv(os.path.join(DATA_PATH, "split_random_val.csv"), header=None),
        "test": pd.read_csv(os.path.join(DATA_PATH, "split_random_test.csv"), header=None),
    },
    "chrono": {
        "train": pd.read_csv(os.path.join(DATA_PATH, "split_chrono_train.csv"), header=None),
        "val": pd.read_csv(os.path.join(DATA_PATH, "split_chrono_val.csv"), header=None),
        "test": pd.read_csv(os.path.join(DATA_PATH, "split_chrono_test.csv"), header=None),
    },
}

SPLIT_TYPE = 'random'
SPLIT_GROUP = 'train'
split = split[SPLIT_TYPE][SPLIT_GROUP]

In [ ]:
def filter_by_split(split_df, df):
    """
    Filter the dataframe based on the split provided.
    
    :param split: The split to filter by (train, val, test)
    :param df: The dataframe to filter
    :return: Filtered dataframe
    """
    
    # for each MRN in the split, get the corresponding rows in the dataframe
    # and return the filtered dataframe
    split_mrns = split_df[0]
    filtered_df = df[df["MRN"].isin(split_mrns)]
    return filtered_df.reset_index(drop=True)

In [ ]:
def one_hot_encode(df, column):
    """
    One-hot encode a specified column in the dataframe.
    
    :param df: The dataframe to encode
    :param column: The column to one-hot encode
    :return: Dataframe with one-hot encoded column
    """
    df[column] = df[column].str.replace(" ", "_")
    # one-hot encode the specified column
    one_hot = pd.get_dummies(df[column], prefix=column)
    one_hot = one_hot.astype(int)
    
    return one_hot

## Raw Features

In [ ]:
# any categorical features here will be one-hot encoded
# more advanced columns like diagnosis or meds will be 
# converted to embeddings later on
raw_features = {
    "visits": [
        "CSN",
        "Visit_no",
        "Age",
        "Triage_Temp",
        "Triage_HR",
        "Triage_RR",
        "Triage_SpO2",
        "Triage_SBP",
        "Triage_DBP",
    ],
}
raw_features_categorical = {
    "visits": [
        "Race",
        "Ethnicity",
        "Means_of_arrival",
        "Gender",
        "Payor_class",
    ]
}

In [ ]:
def extract_raw_features(split_df, feature_type, raw_features_df):
    """
    Extracts the raw features from the dataframe.
    """
    new_features_df = pd.DataFrame()
    split_df = filter_by_split(split_df, raw_features_df)
    
    if feature_type in raw_features:
        for feature in raw_features[feature_type]:
            # keep the numerical features as is
            new_features_df[feature] = split_df[feature]
    if feature_type in raw_features_categorical:
        for feature in raw_features_categorical[feature_type]:
            # one-hot encode the categorical features
            new_features_df = pd.concat([new_features_df, one_hot_encode(split_df, feature)], axis=1)

    return new_features_df

In [ ]:
raw_features_df = extract_raw_features(split, "visits", visits_df)

# save the raw features dataframe to a csv file
raw_features_df.to_csv(os.path.join(DATA_PATH, "features_raw.csv"), index=False)
print(f"{len(raw_features_df.columns)} columns")
print(raw_features_df.columns)

## Engineered features
- visit arrival time (cyclical)
- visit arrival season (categorical)
- shock index (hr bpm / SBP)
- Modified early warning score (MEWS)
- Total medication count (need to use dates for this)

In [ ]:
split_df = filter_by_split(split, visits_df)
engineered_features = split_df[['CSN', 'MRN', 'Arrival_time', 'Departure_time', 'Triage_HR', 'Triage_DBP', 'Triage_SBP', 'Triage_RR', 'Triage_Temp']].copy()

In [ ]:
# print the count of med rows with na entry dates
# Count the number of rows where Entry_date is NaN
na_entry_date_count = meds_df['Entry_date'].isna().sum()
na_start_date_count = meds_df['Start_date'].isna().sum()
na_end_date_count = meds_df['End_date'].isna().sum()
na_arrive_date_count = visits_df['Arrival_time'].isna().sum()
na_depart_date_count = visits_df['Departure_time'].isna().sum()
print(f"Number of rows with NaN Entry_date: {na_entry_date_count}")
print(f"Number of rows with NaN Start_date: {na_start_date_count}")
print(f"Number of rows with NaN End_date: {na_end_date_count}")
print(f"Number of rows with NaN Arrival_time: {na_arrive_date_count}")
print(f"Number of rows with NaN Departure_time: {na_depart_date_count}")


##### Total Medication Count

In [ ]:
from datetime import datetime
import time

def convert_datestr_epoch(datestr):
    """
    Convert a date string to epoch time without using pd.to_datetime, 
    handling ISO 8601 format and dates beyond pandas.Timestamp limits.
    
    :param datestr: The date string to convert (format: YYYY-MM-DDTHH:MM:SSZ)
    :return: The epoch time in seconds
    """
    #print(f"Converting date string: {datestr}")
    if pd.isna(datestr):
        return None
    try:
        # Parse the ISO 8601 date string into a datetime object
        dt = datetime.strptime(datestr, "%Y-%m-%dT%H:%M:%SZ")
        
        # Convert to epoch time in seconds
        epoch_time = int(time.mktime(dt.timetuple()))
        
        return epoch_time
    except ValueError:
        # Handle invalid date strings
        return None

In [ ]:
def get_meds_count(split_df, meds_df):
    """
    Get the count of current home meds for each visit.
    :param split_df: The dataframe with the visits
    :param meds_df: The dataframe with the meds
    :return: Dataframe with the count of home meds for each visit
    """
    # Create a copy to avoid modifying the original dataframe
    result_df = split_df.copy()
    
    # Create a new column for the count of home meds
    result_df["Home_meds_count"] = 0
    
    # Iterate through each row in the split dataframe
    for index, row in result_df.iterrows():
        # Get the MRN for the visit
        mrn = row["MRN"]
        arrival_time = convert_datestr_epoch(row["Arrival_time"])
        
        # Filter the meds dataframe for the MRN
        meds_for_mrn = meds_df[meds_df["MRN"] == mrn]
        
        # Count medications that were current at the time of the visit
        current_meds_count = 0
        for _, med_row in meds_for_mrn.iterrows():
            start_date = convert_datestr_epoch(med_row["Start_date"])
            end_date = convert_datestr_epoch(med_row["End_date"]) if pd.notna(med_row["End_date"]) else float('inf')
            
            # Check if medication was current during the visit
            # (started before arrival and either had no end date or ended after arrival)
            if start_date and start_date <= arrival_time and (not end_date or end_date > arrival_time):
                current_meds_count += 1
        
        # Set the count of home meds for this visit
        result_df.at[index, "Home_meds_count"] = current_meds_count
    
    return result_df

# Get the count of home meds for each visit
engineered_features = get_meds_count(engineered_features, meds_df)
engineered_features.drop(columns=["MRN"], inplace=True)
engineered_features.drop(columns=["Departure_time"], inplace=True)
engineered_features.columns

##### Arrival Time

In [ ]:
# lower year if it is greater than 2262
engineered_features['Arrival_time'] = engineered_features['Arrival_time'].str.replace(r'^.{4}', '2020', regex=True)
engineered_features['Arrival_time'] = pd.to_datetime(engineered_features['Arrival_time'])

# First calculate time in minutes since midnight
minutes_since_midnight = engineered_features['Arrival_time'].dt.hour * 60 + \
                         engineered_features['Arrival_time'].dt.minute


# Convert to radians (full circle = 2π)
time_in_rad = 2 * np.pi * minutes_since_midnight / (24 * 60)

# Create both features
engineered_features['arrival_time_sin'] = np.sin(time_in_rad)
engineered_features['arrival_time_cos'] = np.cos(time_in_rad)

print(engineered_features.columns)

##### Arrival Season

In [ ]:
seasons = {
    1: "winter",
    2: "spring",
    3: "summer",
    4: "fall"
}
arrival_seasons = engineered_features['Arrival_time'].dt.month.map(lambda x: (x % 12 + 3) // 3)
# Convert to seasons
arrival_seasons = arrival_seasons.map(seasons)
# one-hot encode the seasons
arrival_seasons = pd.get_dummies(arrival_seasons, prefix="arrival_time_season").astype(int)
# concatenate the one-hot encoded seasons with the engineered features
engineered_features = pd.concat([engineered_features, arrival_seasons], axis=1)
engineered_features.drop(columns=['Arrival_time'], inplace=True)

engineered_features.columns

##### Shock Index

In [ ]:
engineered_features['shock_index'] = engineered_features['Triage_HR'] / engineered_features['Triage_DBP']
# cap at two decimal places
engineered_features['shock_index'] = engineered_features['shock_index'].round(2)
engineered_features['shock_index'] = engineered_features['shock_index'].replace([np.inf, -np.inf], np.nan)
engineered_features['shock_index'] = engineered_features['shock_index'].fillna(0)
engineered_features['shock_index'] = engineered_features['shock_index'].astype(float)

engineered_features.columns

##### Modified early warning score (MEWS)

In [ ]:
def calculate_mews(sbp, hr, rr, temp, avpu=None):
    """
    Calculate Modified Early Warning Score (MEWS) from individual vital signs.
    
    Parameters:
    sbp (float): Systolic blood pressure in mmHg
    hr (float): Heart rate in beats per minute
    rr (float): Respiratory rate in breaths per minute
    temp (float): Temperature in Celsius
    avpu (str, optional): AVPU score ('A', 'V', 'P', 'U') if available
    
    Returns:
    int: The calculated MEWS score
    """
    # Initialize score
    mews_score = 0
    
    # Systolic BP scoring
    if sbp < 70:
        mews_score += 3
    elif 71 <= sbp <= 80:
        mews_score += 2
    elif 81 <= sbp <= 100:
        mews_score += 1
    elif sbp >= 200:
        mews_score += 2
    
    # Heart rate scoring
    if hr < 40:
        mews_score += 2
    elif 41 <= hr <= 50:
        mews_score += 1
    elif 101 <= hr <= 110:
        mews_score += 1
    elif 111 <= hr <= 129:
        mews_score += 2
    elif hr >= 130:
        mews_score += 3
    
    # Respiratory rate scoring
    if rr < 9:
        mews_score += 2
    elif 15 <= rr <= 20:
        mews_score += 1
    elif 21 <= rr <= 29:
        mews_score += 2
    elif rr >= 30:
        mews_score += 3
    
    # Temperature scoring
    if temp < 35.0:
        mews_score += 2
    elif temp >= 38.5:
        mews_score += 2
    
    # AVPU score if provided
    if avpu:
        if avpu == 'V':  # Responding to Voice
            mews_score += 1
        elif avpu == 'P':  # Responding to Pain
            mews_score += 2
        elif avpu == 'U':  # Unresponsive
            mews_score += 3
    
    return mews_score

In [ ]:
engineered_features['MEWS'] = engineered_features.apply(
    lambda row: calculate_mews(
        row['Triage_SBP'],
        row['Triage_HR'],
        row['Triage_RR'],
        row['Triage_Temp'],
        avpu=None  # Replace with actual AVPU data if available
    ),
    axis=1
)

engineered_features.drop(columns=['Triage_SBP', 'Triage_HR', 'Triage_RR', 'Triage_Temp', 'Triage_DBP'], inplace=True)
engineered_features.columns

In [ ]:
# write the engineered features to a csv file
engineered_features.to_csv(os.path.join(DATA_PATH, "features_engineered.csv"), index=False)

## Combining Raw + Engineered Features

In [ ]:
all_features = pd.merge(raw_features_df, engineered_features, on='CSN', how='inner')
all_features = all_features.drop(columns=['CSN'])
# save the engineered features dataframe to a csv file
all_features.to_csv(os.path.join(DATA_PATH, "features_all.csv"), index=False)
print(f"{len(all_features.columns)} columns")
print(all_features.columns)

## Embedding features
- Visit diagnosis info
    - Dx_name (optional?)
    - CC
- Meds
    - Name
    - Generic_name
    - Med_class
    - Med_subclass
- PMH
    - CodeType (only for prefix)
    - Code
    - Desc10
    - DescCCS

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModel
# import numpy as np

# # Load clinical model
# tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
# model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

# def get_med_embedding(medication_row):
#     # Combine relevant fields
#     text = f"{medication_row['Generic_name']} {medication_row['Med_class']} {medication_row['Med_subclass']}"
    
#     # Generate embedding
#     inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
#     with torch.no_grad():
#         outputs = model(**inputs)
    
#     # Use [CLS] token embedding as representation
#     return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

# # Process patient-level embeddings
# def get_patient_med_embeddings(patient_meds_df):
#     # Get embeddings for each medication
#     med_embeddings = [get_med_embedding(row) for _, row in patient_meds_df.iterrows()]
    
#     # Aggregate (simple mean)
#     if med_embeddings:
#         return np.mean(med_embeddings, axis=0)
#     else:
#         # Return zero vector for patients with no medications
#         return np.zeros((768,))  # Assuming 768-dim embeddings from BERT

In [ ]:
# Target features
# Acuity 1-2
# Acuity 3 with a threshold of ED LOS 1 hr or less where patient was discharged